# Background Voice Detection & Elimination Demo

Background voices can affect age prediction by introducing a second human speaker or TV/radio speech into the audio passed to VoiceAge.

This notebook demonstrates a post-call analysis pipeline that keeps the existing Twilio realtime flow and Wav2Vec2 model logic intact while using **Silero VAD** and **ECAPA-TDNN speaker verification** to produce caller-only audio for age prediction.

## Demo Pipeline

Twilio Call  
->  
Capture Full Caller Audio  
->  
Silero VAD  
->  
Create Caller Reference from First Valid Speech  
->  
ECAPA-TDNN Speaker Verification  
->  
Keep Caller Voice / Reject Background Speakers  
->  
Generate Caller-Only Audio  
->  
Wav2Vec2 Age Prediction  
->  
Generate Final Report


In [ ]:
from __future__ import annotations

import json
import sys
from datetime import UTC, datetime
from pathlib import Path

DEMO_IMPORTS_OK = True
IMPORT_ERRORS = []

try:
    import numpy as np
except Exception as exc:
    DEMO_IMPORTS_OK = False
    IMPORT_ERRORS.append(f"NumPy import failed: {exc}")

try:
    import soundfile as sf
except Exception as exc:
    DEMO_IMPORTS_OK = False
    IMPORT_ERRORS.append(f"soundfile import failed: {exc}")

try:
    from IPython.display import HTML, Markdown, display
except Exception as exc:
    DEMO_IMPORTS_OK = False
    IMPORT_ERRORS.append(f"IPython display import failed: {exc}")
    HTML = None
    Markdown = None
    display = None

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "app").exists():
    for candidate in (PROJECT_ROOT.parent, PROJECT_ROOT.parent.parent):
        if (candidate / "app").exists():
            PROJECT_ROOT = candidate.resolve()
            break

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

try:
    from app.core.config import settings
    from app.services.audio_service import decode_audio_file_full_duration
    from app.services.background_voice_isolation_service import (
        BackgroundVoiceIsolationService,
        SpeechSegment,
        cosine_similarity_score,
    )
    from app.services.model_service import model_service
    from app.services.report_service import write_json
except Exception as exc:
    DEMO_IMPORTS_OK = False
    IMPORT_ERRORS.append(f"Project service import failed: {exc}")


def show_markdown(text: str) -> None:
    if display is not None and Markdown is not None:
        display(Markdown(text))
    else:
        print(text)


def safe_round(value: float | None, digits: int = 3) -> float | None:
    if value is None:
        return None
    return round(float(value), digits)


def overlaps_reference(segment: SpeechSegment, reference_segments: list[SpeechSegment]) -> bool:
    return any(
        segment.start_sample < reference.end_sample and segment.end_sample > reference.start_sample
        for reference in reference_segments
    )


def render_table(rows: list[dict[str, object]], columns: list[str]) -> None:
    if not rows:
        show_markdown("_No rows to display._")
        return

    header_html = "".join(
        f"<th style='text-align:left; padding:6px 10px; border-bottom:1px solid #ccc;'>{column}</th>"
        for column in columns
    )
    body_html = []
    for row in rows:
        cells = "".join(
            f"<td style='padding:6px 10px; border-bottom:1px solid #eee;'>{row.get(column, '')}</td>"
            for column in columns
        )
        body_html.append(f"<tr>{cells}</tr>")

    html = (
        "<table style='border-collapse:collapse; width:100%; font-size:14px;'>"
        f"<thead><tr>{header_html}</tr></thead>"
        f"<tbody>{''.join(body_html)}</tbody>"
        "</table>"
    )
    if display is not None and HTML is not None:
        display(HTML(html))
    else:
        print(rows)


def to_builtin(value):
    if isinstance(value, dict):
        return {str(key): to_builtin(item) for key, item in value.items()}
    if isinstance(value, list):
        return [to_builtin(item) for item in value]
    if isinstance(value, tuple):
        return [to_builtin(item) for item in value]
    if "np" in globals():
        if isinstance(value, np.generic):
            return value.item()
        if isinstance(value, np.ndarray):
            return value.tolist()
    if isinstance(value, Path):
        return str(value)
    return value


full_audio = None
full_audio_duration_sec = None
sample_rate = None
audio_load_error = None
vad_segments = []
vad_table_rows = []
vad_total_speech_duration_sec = 0.0
reference_ready = False
reference_summary = {}
segment_table_rows = []
segment_report_rows = []
kept_segments = []
rejected_segments = []
similarity_values = []
isolated_audio = None
isolated_audio_saved = False
isolated_audio_duration_sec = None
isolation_fallback_reason = None
prediction_payload = None
prediction_error = None


if IMPORT_ERRORS:
    show_markdown(
        "## Setup issue\n"
        "One or more notebook imports failed.\n\n"
        "Run these steps from the repository root and rerun this cell:\n\n"
        "```bash\n"
        "pip install -r requirements.txt\n"
        "jupyter notebook notebooks/background_voice_isolation_demo.ipynb\n"
        "```\n\n"
        "If speaker verification fails on the first run, confirm the machine can download the ECAPA-TDNN model cache.\n\n"
        "**Import errors**\n"
        + "\n".join(f"- {error}" for error in IMPORT_ERRORS)
    )
else:
    show_markdown(
        "## Setup ready\n"
        f"- Project root: `{PROJECT_ROOT}`\n"
        f"- Target sample rate: `{settings.target_sample_rate}` Hz\n"
        f"- Default Wav2Vec2 model path: `{settings.model_path}`\n"
        "- Existing app code stays unchanged; this notebook only reads audio and writes demo outputs."
    )


In [ ]:
if not DEMO_IMPORTS_OK:
    show_markdown("Fix the setup issues above before configuring the demo.")
else:
    realtime_root = PROJECT_ROOT / "data" / "realtime_conversations"
    available_full_audio = []
    if realtime_root.exists():
        available_full_audio = sorted(
            realtime_root.glob("*/caller_full_audio.wav"),
            key=lambda path: path.stat().st_mtime,
            reverse=True,
        )

    default_call_folder = available_full_audio[0].parent if available_full_audio else realtime_root / "CALL_ID_HERE"

    call_folder = default_call_folder
    caller_full_audio_path = call_folder / "caller_full_audio.wav"
    isolated_audio_output_path = call_folder / "caller_full_background_isolated.wav"
    report_output_path = call_folder / "background_voice_isolation_demo_report.json"
    threshold = 0.55
    reference_seconds = 3.0
    min_segment_seconds = 0.30

    show_markdown(
        "## Input configuration\n"
        f"- `call_folder`: `{call_folder}`\n"
        f"- `caller_full_audio.wav`: `{caller_full_audio_path}`\n"
        f"- `isolated output`: `{isolated_audio_output_path}`\n"
        f"- `report output`: `{report_output_path}`\n"
        f"- `threshold`: `{threshold}`\n\n"
        "Edit the variables in this cell if you want to point at a different call folder."
    )

    if available_full_audio:
        recent_examples = [str(path.parent) for path in available_full_audio[:5]]
        show_markdown(
            "Recent call folders with `caller_full_audio.wav`:\n"
            + "\n".join(f"- `{path}`" for path in recent_examples)
        )
    else:
        show_markdown(
            "No existing `caller_full_audio.wav` files were auto-detected. "
            "Update `call_folder` and `caller_full_audio_path` to a valid saved call before proceeding."
        )


In [ ]:
if not DEMO_IMPORTS_OK:
    show_markdown("Fix setup issues first.")
else:
    full_audio = None
    full_audio_duration_sec = None
    sample_rate = None
    audio_load_error = None

    if not caller_full_audio_path.exists():
        audio_load_error = f"Input audio file not found: {caller_full_audio_path}"
        show_markdown(
            "## Input missing\n"
            f"- {audio_load_error}\n\n"
            "Update `caller_full_audio_path` in the configuration cell and rerun."
        )
    else:
        try:
            full_audio = decode_audio_file_full_duration(caller_full_audio_path).astype(np.float32, copy=False)
            sample_rate = settings.target_sample_rate
            full_audio_duration_sec = round(full_audio.size / float(sample_rate), 3)
            show_markdown(
                "## Full caller audio loaded\n"
                f"- Samples: `{full_audio.size}`\n"
                f"- Sample rate: `{sample_rate}` Hz\n"
                f"- Duration: `{full_audio_duration_sec}` seconds"
            )
        except Exception as exc:
            audio_load_error = str(exc)
            show_markdown(
                "## Audio loading failed\n"
                f"- File: `{caller_full_audio_path}`\n"
                f"- Error: `{exc}`\n\n"
                "Confirm that the WAV exists and contains audible caller speech."
            )


In [ ]:
if not DEMO_IMPORTS_OK or full_audio is None:
    show_markdown("Load a valid `caller_full_audio.wav` before running VAD.")
else:
    vad_segments = []
    vad_table_rows = []
    vad_total_speech_duration_sec = 0.0
    isolation_service = BackgroundVoiceIsolationService(
        enabled=True,
        threshold=threshold,
        reference_seconds=reference_seconds,
        min_segment_seconds=min_segment_seconds,
    )

    try:
        vad_segments = isolation_service.detect_speech_segments(full_audio)
        vad_total_speech_duration_sec = round(sum(segment.duration_seconds for segment in vad_segments), 3)
        for index, segment in enumerate(vad_segments, start=1):
            vad_table_rows.append(
                {
                    "segment index": index,
                    "start time": f"{segment.start_sample / float(settings.target_sample_rate):.3f}s",
                    "end time": f"{segment.end_sample / float(settings.target_sample_rate):.3f}s",
                    "duration": f"{segment.duration_seconds:.3f}s",
                }
            )

        show_markdown(
            "## Silero VAD results\n"
            f"- Detected speech segments: `{len(vad_segments)}`\n"
            f"- Total detected speech duration: `{vad_total_speech_duration_sec}` seconds\n"
            f"- Minimum segment length: `{min_segment_seconds}` seconds"
        )
        render_table(vad_table_rows, columns=["segment index", "start time", "end time", "duration"])
    except Exception as exc:
        show_markdown(
            "## VAD failed\n"
            f"- Error: `{exc}`\n\n"
            "Check that `silero-vad` is installed and that the input file contains decodable speech."
        )


In [ ]:
if not DEMO_IMPORTS_OK or full_audio is None or "isolation_service" not in globals():
    show_markdown("Run the setup, configuration, audio load, and VAD cells first.")
else:
    reference_ready = False
    reference_summary = {}

    try:
        reference_ready = isolation_service.initialize_reference(full_audio, vad_segments=vad_segments)
        reference_summary = {
            "reference_ready": bool(reference_ready),
            "reference_source": isolation_service.reference_source,
            "reference_segment_count": isolation_service.reference_segment_count,
            "reference_start_sec": safe_round(isolation_service.reference_start_sec),
            "reference_end_sec": safe_round(isolation_service.reference_end_sec),
            "reference_audio_duration_sec": safe_round(isolation_service.reference_audio_duration_sec),
        }

        if reference_ready:
            show_markdown(
                "## Caller reference created\n"
                f"- Source: `{reference_summary['reference_source']}`\n"
                f"- Reference segments: `{reference_summary['reference_segment_count']}`\n"
                f"- Start: `{reference_summary['reference_start_sec']}` seconds\n"
                f"- End: `{reference_summary['reference_end_sec']}` seconds\n"
                f"- Reference duration: `{reference_summary['reference_audio_duration_sec']}` seconds"
            )
        else:
            show_markdown(
                "## Caller reference not ready\n"
                f"- Source: `{reference_summary['reference_source']}`\n"
                f"- Reported duration: `{reference_summary['reference_audio_duration_sec']}` seconds\n\n"
                "Try lowering `min_segment_seconds` or using a call where the primary caller speaks clearly in the first few seconds."
            )
    except Exception as exc:
        show_markdown(
            "## Reference creation failed\n"
            f"- Error: `{exc}`\n\n"
            "If this is the first run, confirm SpeechBrain can download or access the ECAPA-TDNN model cache."
        )


In [ ]:
if not DEMO_IMPORTS_OK or full_audio is None or "isolation_service" not in globals():
    show_markdown("Run the earlier cells first.")
else:
    segment_table_rows = []
    segment_report_rows = []
    kept_chunks = []
    kept_segments = []
    rejected_segments = []
    similarity_values = []
    isolated_audio = None
    isolation_fallback_reason = None

    if not vad_segments:
        isolation_fallback_reason = "no_speech_segments"
        isolated_audio = full_audio
        show_markdown(
            "## Speaker verification skipped\n"
            "Silero VAD did not find speech segments, so the notebook will fall back to the original full audio."
        )
    elif not reference_ready or isolation_service.reference_embedding is None:
        isolation_fallback_reason = "reference_not_ready"
        isolated_audio = full_audio
        show_markdown(
            "## Speaker verification skipped\n"
            "The caller reference is not ready, so the notebook will fall back to the original full audio."
        )
    else:
        for index, segment in enumerate(vad_segments, start=1):
            segment_audio = full_audio[segment.start_sample:segment.end_sample].astype(np.float32, copy=False)
            start_sec = segment.start_sample / float(settings.target_sample_rate)
            end_sec = segment.end_sample / float(settings.target_sample_rate)
            duration_sec = segment.duration_seconds
            similarity_value = None
            decision = "reject"

            if duration_sec >= min_segment_seconds:
                try:
                    segment_embedding = isolation_service.generate_embedding(segment_audio)
                    similarity_value = float(
                        cosine_similarity_score(isolation_service.reference_embedding, segment_embedding)
                    )
                    similarity_values.append(similarity_value)
                    is_reference_segment = overlaps_reference(segment, isolation_service.reference_segments)
                    decision = "keep" if is_reference_segment or similarity_value >= threshold else "reject"
                except Exception:
                    decision = "reject"

            segment_table_rows.append(
                {
                    "segment index": index,
                    "start time": f"{start_sec:.3f}s",
                    "end time": f"{end_sec:.3f}s",
                    "duration": f"{duration_sec:.3f}s",
                    "similarity": "n/a" if similarity_value is None else f"{similarity_value:.4f}",
                    "decision": decision,
                }
            )
            segment_report_rows.append(
                {
                    "segment_index": index,
                    "start_time_sec": round(start_sec, 3),
                    "end_time_sec": round(end_sec, 3),
                    "duration_sec": round(duration_sec, 3),
                    "similarity": round(similarity_value, 4) if similarity_value is not None else None,
                    "decision": decision,
                }
            )

            if decision == "keep":
                kept_chunks.append(segment_audio)
                kept_segments.append(segment)
            else:
                rejected_segments.append(segment)

        if kept_chunks:
            isolated_audio = np.concatenate(kept_chunks).astype(np.float32)
        else:
            isolated_audio = full_audio
            isolation_fallback_reason = "all_segments_rejected"

        avg_similarity = round(sum(similarity_values) / len(similarity_values), 4) if similarity_values else None
        show_markdown("## Speaker verification decisions")
        render_table(
            segment_table_rows,
            columns=["segment index", "start time", "end time", "duration", "similarity", "decision"],
        )
        show_markdown(
            f"- Kept segments: `{len(kept_segments)}`\n"
            f"- Rejected segments: `{len(rejected_segments)}`\n"
            f"- Average similarity: `{avg_similarity if avg_similarity is not None else 'n/a'}`\n"
            f"- Fallback used: `{'yes' if isolation_fallback_reason else 'no'}`"
        )


In [ ]:
if not DEMO_IMPORTS_OK or full_audio is None:
    show_markdown("Load input audio first.")
else:
    isolated_audio_saved = False
    isolated_audio_duration_sec = None

    if "isolated_audio" not in globals() or isolated_audio is None:
        show_markdown("Run the speaker verification cell first to generate isolated audio.")
    else:
        try:
            call_folder.mkdir(parents=True, exist_ok=True)
            sf.write(isolated_audio_output_path, isolated_audio, settings.target_sample_rate)
            isolated_audio_saved = True
            isolated_audio_duration_sec = round(isolated_audio.size / float(settings.target_sample_rate), 3)
            show_markdown(
                "## Isolated audio saved\n"
                f"- Output: `{isolated_audio_output_path}`\n"
                f"- Duration: `{isolated_audio_duration_sec}` seconds"
            )
        except Exception as exc:
            show_markdown(
                "## Failed to save isolated audio\n"
                f"- Output: `{isolated_audio_output_path}`\n"
                f"- Error: `{exc}`"
            )


In [ ]:
if not DEMO_IMPORTS_OK:
    show_markdown("Fix setup issues first.")
else:
    prediction_payload = None
    prediction_error = None

    if not globals().get("isolated_audio_saved", False):
        show_markdown("Save the isolated audio before running Wav2Vec2 prediction.")
    else:
        try:
            model_service.load_model()
            prediction = model_service.predict_audio_file(isolated_audio_output_path)
            if hasattr(prediction, "model_dump"):
                prediction_payload = prediction.model_dump()
            else:
                prediction_payload = prediction.dict()

            show_markdown(
                "## Wav2Vec2 age prediction\n"
                f"- Predicted age group: `{prediction_payload['predicted_age_group']}`\n"
                f"- Confidence: `{prediction_payload['confidence']:.4f}`\n"
                f"- Confidence level: `{prediction_payload['confidence_level']}`\n"
                f"- Model version: `{prediction_payload['model_version']}`"
            )
        except Exception as exc:
            prediction_error = str(exc)
            show_markdown(
                "## Wav2Vec2 prediction unavailable\n"
                f"- Error: `{exc}`\n\n"
                "The isolation demo is still valid. If you want the final age prediction, confirm the model path exists and the service can load the current Wav2Vec2 artifacts."
            )


In [ ]:
if not DEMO_IMPORTS_OK or full_audio is None:
    show_markdown("Load input audio first.")
else:
    if "isolated_audio" not in globals() or isolated_audio is None:
        show_markdown("Run the speaker verification cell first to build the final report.")
    else:
        isolated_audio_duration_sec = round(isolated_audio.size / float(settings.target_sample_rate), 3)
        avg_similarity = round(sum(similarity_values) / len(similarity_values), 4) if similarity_values else None
        report_payload = {
            "generated_at_utc": datetime.now(UTC).isoformat(),
            "pipeline": [
                "Twilio Call",
                "Capture Full Caller Audio",
                "Silero VAD",
                "Create Caller Reference from First Valid Speech",
                "ECAPA-TDNN Speaker Verification",
                "Keep Caller Voice / Reject Background Speakers",
                "Generate Caller-Only Audio",
                "Wav2Vec2 Age Prediction",
                "Generate Final Report",
            ],
            "call_folder": str(call_folder),
            "input_audio_path": str(caller_full_audio_path),
            "isolated_audio_path": str(isolated_audio_output_path),
            "threshold": threshold,
            "reference_seconds": reference_seconds,
            "min_segment_seconds": min_segment_seconds,
            "sample_rate": settings.target_sample_rate,
            "full_audio_duration_sec": full_audio_duration_sec,
            "isolated_audio_duration_sec": isolated_audio_duration_sec,
            "vad_segments": len(vad_segments),
            "kept_segments": len(kept_segments),
            "rejected_segments": len(rejected_segments),
            "fallback_used": isolation_fallback_reason is not None,
            "fallback_reason": isolation_fallback_reason,
            "reference": {
                "ready": bool(reference_ready),
                "source": isolation_service.reference_source,
                "segment_count": isolation_service.reference_segment_count,
                "start_sec": safe_round(isolation_service.reference_start_sec),
                "end_sec": safe_round(isolation_service.reference_end_sec),
                "duration_sec": safe_round(isolation_service.reference_audio_duration_sec),
            },
            "verification": {
                "avg_similarity": avg_similarity,
                "threshold": threshold,
                "segment_results": segment_report_rows,
            },
            "prediction": prediction_payload,
            "prediction_error": prediction_error,
        }

        try:
            write_json(report_output_path, to_builtin(report_payload))
            show_markdown(
                "## Final report saved\n"
                f"- JSON report: `{report_output_path}`\n"
                f"- Audio output: `{isolated_audio_output_path}`"
            )
        except Exception as exc:
            show_markdown(
                "## Failed to save JSON report\n"
                f"- Output: `{report_output_path}`\n"
                f"- Error: `{exc}`"
            )


In [ ]:
if not DEMO_IMPORTS_OK or full_audio is None:
    show_markdown("Load input audio first.")
else:
    predicted_age_group = prediction_payload["predicted_age_group"] if prediction_payload else "unavailable"
    confidence_value = f"{prediction_payload['confidence']:.4f}" if prediction_payload else "unavailable"

    summary_rows = [
        {"metric": "full audio duration", "value": f"{full_audio_duration_sec:.3f}s" if full_audio_duration_sec is not None else "n/a"},
        {"metric": "VAD segments", "value": len(vad_segments)},
        {"metric": "kept segments", "value": len(kept_segments)},
        {"metric": "rejected segments", "value": len(rejected_segments)},
        {"metric": "threshold", "value": f"{threshold:.2f}"},
        {"metric": "predicted age group", "value": predicted_age_group},
        {"metric": "confidence", "value": confidence_value},
    ]

    show_markdown("## Final summary")
    render_table(summary_rows, columns=["metric", "value"])

    if prediction_error:
        show_markdown(f"Prediction note: `{prediction_error}`")


## Demo Notes

- Tune `threshold` based on reference quality and the amount of background speech.
- Start testing in the `0.40` to `0.65` range, then tighten if too much background speech is kept.
- This is a segment filter, not true source separation.
- Overlapping speakers may fail because both voices can exist inside the same VAD segment.
- Very short or noisy early caller speech can produce a weak reference embedding.
- On the first run, ECAPA-TDNN model artifacts may need to download before speaker verification succeeds.
